In [7]:
import pandas as pd
import glob
import os

In [8]:
import pandas as pd

# 1. Încarcă fișierele
df_pomi = pd.read_csv('nr_pomi.csv')
df_total = pd.read_csv('tone_totale.csv')
df_medie = pd.read_csv('kg_pom.csv')

# 2. Redenumește coloanele ca să scapi de numele lungi de la INS
# Folosim .columns[index] pentru a nu mai scrie tot textul ala lung cu Macroregiuni
def curata_df(df, nume_valoare):
    # Stergem spatiile goale din numele coloanelor
    df.columns = df.columns.str.strip()
    
    # Redenumim coloanele standard (bazat pe pozitia lor in fisierul INS)
    # De obicei: [0]=Categorie, [1]=Proprietate, [2]=Locatie, [3]=An, [4]=UM, [5]=Valoare
    df.rename(columns={
        'Ani': 'An',
        'Macroregiuni  regiuni de dezvoltare si judete': 'Judet', # Atentie la dublu spatiu aici!
        'Valoare': nume_valoare
    }, inplace=True)
    
    # Daca numele lung tot face figuri, fortam redenumirea dupa pozitie:
    df.columns.values[2] = 'Judet' 
    return df

df_pomi = curata_df(df_pomi, 'Nr_Pomi')
df_total = curata_df(df_total, 'Tone_Totale')
df_medie = curata_df(df_medie, 'Kg_per_Pom')

# 3. Merge-ul devine mult mai simplu acum
df_master = pd.merge(df_pomi, df_total, on=['An', 'Judet'])
df_master = pd.merge(df_master, df_medie, on=['An', 'Judet'])

# 4. Salvează
df_master.to_csv('master_data.csv', index=False)
print("Merge reusit! Fisierul master_data.csv a fost creat.")

Merge reusit! Fisierul master_data.csv a fost creat.


In [10]:
df_master = df_master.rename(columns={
    'Categorii de pomi fructiferi': 'Specie_Pomi',
})

df_master= df_master[['An', 'Judet', 'Specie_Pomi', 'Nr_Pomi', 'Tone_Totale', 'Kg_per_Pom']]
df_master.to_csv('master_data.csv', index=False)

In [12]:
df_master['Max_Kg_Istoric'] = df_master.groupby('Judet')['Kg_per_Pom'].transform('max')

# 2. Modificăm funcția să se uite la acest potențial maxim
def identifica_rootstock_stabil(row):
    potenial_maxim = row['Max_Kg_Istoric']
    
    if potenial_maxim < 20:   # Pragurile pot fi ajustate
        return 1  # Pitic (M9)
    elif potenial_maxim < 38:
        return 2  # Semipitic (M26)
    elif potenial_maxim < 65:
        return 3  # Semiviguros (MM106)
    else:
        return 4  # Viguros (Salbatic)

# 3. Aplicăm pe coloana de potențial
df_master['Rootstock_ID'] = df_master.apply(identifica_rootstock_stabil, axis=1)

print(df_master.head())

           An   Judet Specie_Pomi  Nr_Pomi  Tone_Totale  Kg_per_Pom  \
0   Anul 2000   Bihor       Pruni  1563587        15430          10   
1   Anul 2000   Bihor       Pruni  1563587        15430           7   
2   Anul 2000   Bihor       Pruni  1563587        15430           9   
3   Anul 2000   Bihor       Pruni  1563587        15430           9   
4   Anul 2000   Bihor       Pruni  1563587        15430           8   

   Max_Kg_Istoric  Rootstock_ID  
0              45             3  
1              45             3  
2              45             3  
3              45             3  
4              45             3  
